In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import * 
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("sample_app") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# S3 bucket details, access key, secret keys should be in environmental variable
# For s3 access
spark._jsc.hadoopConfiguration().set(
    "fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark._jsc.hadoopConfiguration().set(
    "fs.s3a.endpoint", "s3.amazonaws.com")

spark.sparkContext.setLogLevel("WARN")


#### Use of multiple spark session
#### 1. To test your logic in other session
#### 2. globalTempView will be available for multiple spark session

In [142]:
spark_session2 = spark.newSession()
print("session1", spark, "\nsession2", spark_session2)

session1 <pyspark.sql.session.SparkSession object at 0x792f45334c90> 
session2 <pyspark.sql.session.SparkSession object at 0x792fbc100f90>


#### Defining Schema
1. Schema DDL
2. structType and nested schema

In [311]:
from datetime import date

data = [(1, "a", date(2020, 1, 1))]
schema = 'id int, name string, dob date'
df = spark.createDataFrame(data, schema)
df.show()

+---+----+----------+
| id|name|       dob|
+---+----+----------+
|  1|   a|2020-01-01|
+---+----+----------+



In [317]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

schema = StructType([StructField("id", IntegerType()),
                    StructField("name", StringType(), True),
                    StructField("dob", DateType(), True),
                   StructField("address", StructType([StructField("city", StringType(), True),
                                                     StructField("zip", IntegerType(), True)]))])
data = [(1, "abc", date(2020, 1, 1), ("Blore", 560064))]
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+---+----+----------+---------------+
| id|name|       dob|        address|
+---+----+----------+---------------+
|  1| abc|2020-01-01|{Blore, 560064}|
+---+----+----------+---------------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: integer (nullable = true)



In [320]:
# fillna
# data validations
# data quality checks

#### Read and Write to S3 and JDBC
#### Files Write modes: APPEND, OVERWRITE, ERROR, IGNORE

In [170]:
s3_read_df = spark.read.format("parquet").option("header", True).option("inferSchema", True).option("samplingRation", 0.5).load("s3a://dbt-snow-bucket/emp")
# It’s not recommended to infer the schema 
# •	as spark has to scan data to determine datatypes, so it is time consuming
# •	it might infer incorrectly (eg. order_date)
# If you still want to inter schema, we can do using sample data
# 0.5 => 50%  0.01 => 1%

s3_read_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- promoted: boolean (nullable = true)
 |-- state: string (nullable = true)
 |-- date_modified: timestamp (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = true)



In [145]:
s3_read_df.write.format("parquet").mode("append").save("s3a://dbt-snow-bucket/emp_save")  # Another part-0000 file will be added in same path
# s3_read_df.write.format("parquet").mode("overwrite").save("s3a://dbt-snow-bucket/emp_save")  # Overwrites files
# s3_read_df.write.format("parquet").mode("error").save("s3a://dbt-snow-bucket/emp_save")  # PATH_ALREADY_EXISTS error
# s3_read_df.write.format("parquet").mode("ignore").save("s3a://dbt-snow-bucket/emp_save")  # Error will not be thrown, data will be written only if foler is empty

In [161]:
import builtins

jdbc_url = "jdbc:mysql://mysql_container:3306/employees"
user = 'root'
password = 'root'
driver = 'com.mysql.cj.jdbc.Driver'
fetchsize = 1000
table = 'src_emp'
partition_column = 'emp_id'
num_partitions=10

# fetchsize=1000 num_partitions=10 👉 Means:
# Spark creates 10 parallel tasks (partitions)
# Each task opens 1 JDBC connection
# Each connection fetches data in chunks of 1000 rows per round trip

def base_reader():
    return (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", user)
        .option("password", password)
        .option("driver", driver)
        .option("fetchsize", fetchsize)
    )
        
# Get the upper and lower bound
bounds_query = f"""
    (SELECT 
        MIN({partition_column}) AS min_val,
        MAX({partition_column}) AS max_val,
        COUNT(*) AS total_count
     FROM {table}) AS bounds
"""
bounds = (
    base_reader()
    .option("dbtable", bounds_query)
    .load()
    .first()
)

lower = bounds["min_val"]
upper = bounds["max_val"]
total = bounds["total_count"]

if total < num_partitions:
    num_partitions = builtins.max(1, int(total))
        
new_df = base_reader()\
    .option("dbtable", f"""
        (SELECT * FROM {table} 
         WHERE date_modified > '{last_read}') AS inc """)\
    .option("partitionColumn", partition_column)\
    .option("lowerBound", lower)\
    .option("upperBound", upper)\
    .option("numPartitions", num_partitions)\
    .load()
new_df.show()

+------+----------+------+--------+---------+-------------------+
|emp_id|      name|salary|promoted|    state|      date_modified|
+------+----------+------+--------+---------+-------------------+
|     1|Ravi Kumar| 55000|   false|Karnataka|2026-04-06 00:00:00|
|     9|     Hegde|255000|    true|Karnataka|2026-04-06 00:00:00|
+------+----------+------+--------+---------+-------------------+



In [172]:
new_df = new_df.repartition(8)  # Is like option("num_partitions", 8)

new_df.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "write_emp")\
    .option("user", user)\
    .option("password", password)\
    .option("driver", driver)\
    .option("batchsize", 5000)\
    .mode("append")\
    .option("truncate", False)\
    .option("isolationLevel", "READ_COMMITTED").save()

# | Use case                  | isolationLevel    |
# | ------------------------- | ----------------- |
# | General ETL               | `READ_COMMITTED`  |
# | High consistency required | `REPEATABLE_READ` |
# | Rare strict correctness   | `SERIALIZABLE`    |


#### Simple aggregation: n inputs => Output row 1
#### Grouping: n group => n output row
#### Window Functions - ROW_NUM(), RANK(), DENSE_RANK(), LEAD, LAG, Running total

In [ ]:
# CREATE TABLE sales_data (
#     id INT PRIMARY KEY,
#     salesperson VARCHAR(50),
#     region VARCHAR(50),
#     sale_date DATE,
#     amount INT
# );

# INSERT INTO sales_data (id, salesperson, region, sale_date, amount) VALUES
# (1, 'Amit', 'North', '2024-01-01', 500),
# (2, 'Amit', 'North', '2024-01-05', 700),
# (3, 'Amit', 'North', '2024-01-10', 700),
# (4, 'Ravi', 'South', '2024-01-02', 400),
# (5, 'Ravi', 'South', '2024-01-06', 600),
# (6, 'Ravi', 'South', '2024-01-11', 600),
# (7, 'Neha', 'West', '2024-01-03', 300),
# (8, 'Neha', 'West', '2024-01-07', 800),
# (9, 'Neha', 'West', '2024-01-12', 800);
# INSERT INTO sales_data (id, salesperson, region, sale_date, amount) VALUES
# (10, 'Amit', 'North', '2024-01-15', 900),
# (11, 'Amit', 'North', '2024-01-20', 300),
# (12, 'Amit', 'North', '2024-01-25', 400),
# (13, 'Ravi', 'South', '2024-01-15', 500),
# (14, 'Ravi', 'South', '2024-01-20', 700),
# (15, 'Ravi', 'South', '2024-01-25', 200),
# (16, 'Neha', 'West', '2024-01-15', 600),
# (17, 'Neha', 'West', '2024-01-20', 100),
# (18, 'Neha', 'West', '2024-01-25', 900);

In [2]:
jdbc_url = "jdbc:mysql://mysql_container:3306/employees"
input_df = spark.read.format("jdbc").option("url", jdbc_url)\
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", f"""
        (SELECT * FROM sales_data) AS inc """)\
    .option("user", "root") \
    .option("password", "root").load()
input_df.show(5)

+---+-----------+------+----------+------+
| id|salesperson|region| sale_date|amount|
+---+-----------+------+----------+------+
|  1|       Amit| North|2024-01-01|   500|
|  2|       Amit| North|2024-01-05|   700|
|  3|       Amit| North|2024-01-10|   700|
|  4|       Ravi| South|2024-01-02|   400|
|  5|       Ravi| South|2024-01-06|   600|
+---+-----------+------+----------+------+
only showing top 5 rows



In [186]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.amount)
output1_df = input_df.withColumn("row_number", row_number().over(window)).withColumn("rank", rank().over(window)).withColumn("dense_rank", dense_rank().over(window))
output1_df.select("salesperson", "amount", "row_number", "rank", "dense_rank").show()

+-----------+------+----------+----+----------+
|salesperson|amount|row_number|rank|dense_rank|
+-----------+------+----------+----+----------+
|       NULL|   900|         1|   1|         1|
|       Amit|   300|         1|   1|         1|
|       Amit|   400|         2|   2|         2|
|       Amit|   500|         3|   3|         3|
|       Amit|   700|         4|   4|         4|
|       Amit|   700|         5|   4|         4|
|       Amit|   900|         6|   6|         5|
|       Neha|   100|         1|   1|         1|
|       Neha|   300|         2|   2|         2|
|       Neha|   600|         3|   3|         3|
|       Neha|   800|         4|   4|         4|
|       Neha|   800|         5|   4|         4|
|       Neha|   900|         6|   6|         5|
|       Ravi|   200|         1|   1|         1|
|       Ravi|   400|         2|   2|         2|
|       Ravi|   500|         3|   3|         3|
|       Ravi|   600|         4|   4|         4|
|       Ravi|   600|         5|   4|    

In [187]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date)
output_df2 = input_df.withColumn("lead", lead(input_df.amount).over(window)).withColumn("lag", lag(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "lead", "lag").show()

+-----------+----------+------+----+----+
|salesperson| sale_date|amount|lead| lag|
+-----------+----------+------+----+----+
|       NULL|2024-01-15|   900|NULL|NULL|
|       Amit|2024-01-01|   500| 700|NULL|
|       Amit|2024-01-05|   700| 700| 500|
|       Amit|2024-01-10|   700| 900| 700|
|       Amit|2024-01-15|   900| 300| 700|
|       Amit|2024-01-20|   300| 400| 900|
|       Amit|2024-01-25|   400|NULL| 300|
|       Neha|2024-01-03|   300| 800|NULL|
|       Neha|2024-01-07|   800| 800| 300|
|       Neha|2024-01-12|   800| 600| 800|
|       Neha|2024-01-15|   600| 100| 800|
|       Neha|2024-01-20|   100| 900| 600|
|       Neha|2024-01-25|   900|NULL| 100|
|       Ravi|2024-01-02|   400| 600|NULL|
|       Ravi|2024-01-06|   600| 600| 400|
|       Ravi|2024-01-11|   600| 500| 600|
|       Ravi|2024-01-15|   500| 700| 600|
|       Ravi|2024-01-20|   700| 200| 500|
|       Ravi|2024-01-25|   200|NULL| 700|
+-----------+----------+------+----+----+



In [188]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date).rowsBetween(Window.unboundedPreceding, Window.currentRow)
output_df2 = input_df.withColumn("sum", sum(input_df.amount).over(window)).withColumn("avg", avg(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "sum", "avg").show()

+-----------+----------+------+----+-----------------+
|salesperson| sale_date|amount| sum|              avg|
+-----------+----------+------+----+-----------------+
|       NULL|2024-01-15|   900| 900|            900.0|
|       Amit|2024-01-01|   500| 500|            500.0|
|       Amit|2024-01-05|   700|1200|            600.0|
|       Amit|2024-01-10|   700|1900|633.3333333333334|
|       Amit|2024-01-15|   900|2800|            700.0|
|       Amit|2024-01-20|   300|3100|            620.0|
|       Amit|2024-01-25|   400|3500|583.3333333333334|
|       Neha|2024-01-03|   300| 300|            300.0|
|       Neha|2024-01-07|   800|1100|            550.0|
|       Neha|2024-01-12|   800|1900|633.3333333333334|
|       Neha|2024-01-15|   600|2500|            625.0|
|       Neha|2024-01-20|   100|2600|            520.0|
|       Neha|2024-01-25|   900|3500|583.3333333333334|
|       Ravi|2024-01-02|   400| 400|            400.0|
|       Ravi|2024-01-06|   600|1000|            500.0|
|       Ra

In [189]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date).rowsBetween(-2, Window.currentRow)  # Sum of last 3 rows 
output_df2 = input_df.withColumn("sum", sum(input_df.amount).over(window)).withColumn("avg", avg(input_df.amount).over(window))
output_df2.select("salesperson", "sale_date", "amount", "sum", "avg").show()

+-----------+----------+------+----+-----------------+
|salesperson| sale_date|amount| sum|              avg|
+-----------+----------+------+----+-----------------+
|       NULL|2024-01-15|   900| 900|            900.0|
|       Amit|2024-01-01|   500| 500|            500.0|
|       Amit|2024-01-05|   700|1200|            600.0|
|       Amit|2024-01-10|   700|1900|633.3333333333334|
|       Amit|2024-01-15|   900|2300|766.6666666666666|
|       Amit|2024-01-20|   300|1900|633.3333333333334|
|       Amit|2024-01-25|   400|1600|533.3333333333334|
|       Neha|2024-01-03|   300| 300|            300.0|
|       Neha|2024-01-07|   800|1100|            550.0|
|       Neha|2024-01-12|   800|1900|633.3333333333334|
|       Neha|2024-01-15|   600|2200|733.3333333333334|
|       Neha|2024-01-20|   100|1500|            500.0|
|       Neha|2024-01-25|   900|1600|533.3333333333334|
|       Ravi|2024-01-02|   400| 400|            400.0|
|       Ravi|2024-01-06|   600|1000|            500.0|
|       Ra

#### count, count(col_name), countDistinct(col_name) excludes NULL
#### count(*) includes NULL

In [190]:
input_df.select(count("*"), count(input_df.salesperson), countDistinct(input_df.salesperson)).show()  
input_df.select(input_df.salesperson).distinct().show()

+--------+------------------+---------------------------+
|count(1)|count(salesperson)|count(DISTINCT salesperson)|
+--------+------------------+---------------------------+
|      19|                18|                          3|
+--------+------------------+---------------------------+

+-----------+
|salesperson|
+-----------+
|       Ravi|
|       Amit|
|       Neha|
|       NULL|
+-----------+



In [191]:
window = Window.partitionBy(input_df.salesperson).orderBy(input_df.sale_date)
output_df3 = input_df.withColumn("sum", sum(input_df.amount).over(window))
output_df3.select("salesperson", "sale_date", "amount", "sum").show(10)

# Without orderBy, Aggregation column will be same for all rows in each group
window1 = Window.partitionBy(input_df.salesperson)
output_df3 = input_df.withColumn("sum", sum(input_df.amount).over(window1))
output_df3.select("salesperson", "sale_date", "amount", "sum").show(10)

+-----------+----------+------+----+
|salesperson| sale_date|amount| sum|
+-----------+----------+------+----+
|       NULL|2024-01-15|   900| 900|
|       Amit|2024-01-01|   500| 500|
|       Amit|2024-01-05|   700|1200|
|       Amit|2024-01-10|   700|1900|
|       Amit|2024-01-15|   900|2800|
|       Amit|2024-01-20|   300|3100|
|       Amit|2024-01-25|   400|3500|
|       Neha|2024-01-03|   300| 300|
|       Neha|2024-01-07|   800|1100|
|       Neha|2024-01-12|   800|1900|
+-----------+----------+------+----+
only showing top 10 rows

+-----------+----------+------+----+
|salesperson| sale_date|amount| sum|
+-----------+----------+------+----+
|       NULL|2024-01-15|   900| 900|
|       Amit|2024-01-01|   500|3500|
|       Amit|2024-01-05|   700|3500|
|       Amit|2024-01-10|   700|3500|
|       Amit|2024-01-15|   900|3500|
|       Amit|2024-01-20|   300|3500|
|       Amit|2024-01-25|   400|3500|
|       Neha|2024-01-03|   300|3500|
|       Neha|2024-01-07|   800|3500|
|       Neha

#### Some Transformations
* withColumn
* withColumnRenamed
* drop
* dropDuplicates
* case-when
* filterpivot
* isin
* isnotin => ~(isin)
* isNull()
* isNotNull()
* collect_list
* collect_set
* split by multiple char
* explode
* explode_outer
* posexplode
* date_format
* array_contains

In [206]:
transformed_df = input_df.withColumn("withColumn", lit('sample')).withColumn("oldColumn", lit("addColumn"))
transformed_df = transformed_df.withColumnRenamed("oldColumn", "withColumnRenamed").withColumn("casting", to_timestamp("sale_date"))
transformed_df.filter(transformed_df.region == 'North')
transformed_df = transformed_df.withColumn("case_when", when(transformed_df.region=='North', 'N')\
                                           .when(transformed_df.region=='South', 'S')\
                                           .when(transformed_df.region=='West', 'W')\
                                           .otherwise('E'))
transformed_df = transformed_df.drop(transformed_df.salesperson)
distinct_drop_df = transformed_df.select(transformed_df.id, transformed_df.region, transformed_df.amount)
distinct_drop_df.distinct().orderBy(distinct_drop_df.id).show()
distinct_drop_df.dropDuplicates().orderBy(distinct_drop_df.id).show()  # All columns are compared
distinct_drop_df.dropDuplicates(['region', 'amount']).orderBy(distinct_drop_df.id).show()

# distinct                              | dropduplicates
# Gives distinct vales in all columns	| Can specify which column to consider


+----+------+------+
|  id|region|amount|
+----+------+------+
|   1| North|   500|
|   2| North|   700|
|   3| North|   700|
|   4| South|   400|
|   5| South|   600|
|   6| South|   600|
|   7|  West|   300|
|   8|  West|   800|
|   9|  West|   800|
|  10| North|   900|
|  11| North|   300|
|  12| North|   400|
|  13| South|   500|
|  14| South|   700|
|  15| South|   200|
|  16|  West|   600|
|  17|  West|   100|
|  18|  West|   900|
|9999| North|   900|
+----+------+------+

+---+------+------+
| id|region|amount|
+---+------+------+
|  1| North|   500|
|  2| North|   700|
|  4| South|   400|
|  5| South|   600|
|  7|  West|   300|
|  8|  West|   800|
| 10| North|   900|
| 11| North|   300|
| 12| North|   400|
| 13| South|   500|
| 14| South|   700|
| 15| South|   200|
| 16|  West|   600|
| 17|  West|   100|
| 18|  West|   900|
+---+------+------+



#### Pivot
* GroupBy: Which column should be vertical. That is rows
* Pivot: Which column should be horrizontal. That is columns
* Aggregation function: What agg to perform. That is value

In [218]:
pivot_df = transformed_df.select('region', 'sale_date', 'amount').groupBy(transformed_df.sale_date).pivot('region').agg(sum(transformed_df.amount))
pivot_df.show()
# Verify result
transformed_df.filter((transformed_df.sale_date == '2024-01-15') & (transformed_df.region == 'North')).show()

+----------+-----+-----+----+
| sale_date|North|South|West|
+----------+-----+-----+----+
|2024-01-07| NULL| NULL| 800|
|2024-01-11| NULL|  600|NULL|
|2024-01-02| NULL|  400|NULL|
|2024-01-05|  700| NULL|NULL|
|2024-01-15| 1800|  500| 600|
|2024-01-10|  700| NULL|NULL|
|2024-01-12| NULL| NULL| 800|
|2024-01-03| NULL| NULL| 300|
|2024-01-20|  300|  700| 100|
|2024-01-06| NULL|  600|NULL|
|2024-01-01|  500| NULL|NULL|
|2024-01-25|  400|  200| 900|
+----------+-----+-----+----+

+----+------+----------+------+----------+-----------------+-------------------+-----------------+---------+
|  id|region| sale_date|amount|withColumn|withColumnRenamed|            casting|withColumnRenamed|case_when|
+----+------+----------+------+----------+-----------------+-------------------+-----------------+---------+
|  10| North|2024-01-15|   900|    sample|        addColumn|2024-01-15 00:00:00|        addColumn|        N|
|9999| North|2024-01-15|   900|    sample|        addColumn|2024-01-15 00:00:00|   

In [220]:
# To optimize
region_list = ['North', 'South', 'West', 'East']
pivot_df = transformed_df.groupBy(transformed_df.sale_date).pivot('region', region_list).agg(sum(transformed_df.amount))
pivot_df.show()

+----------+-----+-----+----+----+
| sale_date|North|South|West|East|
+----------+-----+-----+----+----+
|2024-01-07| NULL| NULL| 800|NULL|
|2024-01-11| NULL|  600|NULL|NULL|
|2024-01-02| NULL|  400|NULL|NULL|
|2024-01-05|  700| NULL|NULL|NULL|
|2024-01-15| 1800|  500| 600|NULL|
|2024-01-10|  700| NULL|NULL|NULL|
|2024-01-12| NULL| NULL| 800|NULL|
|2024-01-03| NULL| NULL| 300|NULL|
|2024-01-20|  300|  700| 100|NULL|
|2024-01-06| NULL|  600|NULL|NULL|
|2024-01-01|  500| NULL|NULL|NULL|
|2024-01-25|  400|  200| 900|NULL|
+----------+-----+-----+----+----+



In [232]:
transformed_df.filter(transformed_df.region.isin(['South'])).select('region').distinct().show()
transformed_df.filter(~transformed_df.region.isin(['South'])).select('region').distinct().show()

pivot_df.filter(pivot_df.North.isNull()).select('North').distinct().show()
pivot_df.filter(pivot_df.North.isNotNull()).select('North').show()

+------+
|region|
+------+
| South|
+------+

+------+
|region|
+------+
|  West|
| North|
+------+

+-----+
|North|
+-----+
| NULL|
+-----+

+-----+
|North|
+-----+
|  700|
| 1800|
|  700|
|  300|
|  500|
|  400|
+-----+



In [238]:
transformed_df.groupBy(transformed_df.region)\
    .agg(collect_list(transformed_df.amount).alias("collect_list")).show(2, False)
transformed_df.groupBy(transformed_df.region)\
    .agg(collect_set(transformed_df.amount).alias("collect_list")).show(2, False)

+------+------------------------------+
|region|collect_list                  |
+------+------------------------------+
|South |[400, 600, 600, 500, 700, 200]|
|West  |[300, 800, 800, 600, 100, 900]|
+------+------------------------------+
only showing top 2 rows

+------+-------------------------+
|region|collect_list             |
+------+-------------------------+
|South |[700, 500, 600, 400, 200]|
|West  |[300, 800, 100, 900, 600]|
+------+-------------------------+
only showing top 2 rows



In [277]:
data = [(None, None), ("chaitra,hegde", 30), ("shiv-hegde", 2), ("shubh,g-hegde", 2), ("abc,gouda", 5)]
schema = "name string, age int"
df = spark.createDataFrame(data, schema)
df.show()
df.withColumn("explode", explode(split(col("name"), "[,-]"))).show()  # In single line
df = df.withColumn("split_name", split(col("name"), "[,-]"))  # Multiple splits
df.withColumn("explode", explode(col("split_name"))).show()  # Null re excluded
df.withColumn("explode_outer", explode_outer(col("split_name"))).show()  # Null re excluded
df.select("*", posexplode(col("split_name")).alias("pos", "posexplode")).show()  # WithColumn wont work here because posexplode returns 2 column - pos and value
df.withColumn("array_contains", array_contains(col("split_name"), "hegde")).show()

+-------------+----+
|         name| age|
+-------------+----+
|         NULL|NULL|
|chaitra,hegde|  30|
|   shiv-hegde|   2|
|shubh,g-hegde|   2|
|    abc,gouda|   5|
+-------------+----+

+-------------+---+-------+
|         name|age|explode|
+-------------+---+-------+
|chaitra,hegde| 30|chaitra|
|chaitra,hegde| 30|  hegde|
|   shiv-hegde|  2|   shiv|
|   shiv-hegde|  2|  hegde|
|shubh,g-hegde|  2|  shubh|
|shubh,g-hegde|  2|      g|
|shubh,g-hegde|  2|  hegde|
|    abc,gouda|  5|    abc|
|    abc,gouda|  5|  gouda|
+-------------+---+-------+

+-------------+---+-----------------+-------+
|         name|age|       split_name|explode|
+-------------+---+-----------------+-------+
|chaitra,hegde| 30| [chaitra, hegde]|chaitra|
|chaitra,hegde| 30| [chaitra, hegde]|  hegde|
|   shiv-hegde|  2|    [shiv, hegde]|   shiv|
|   shiv-hegde|  2|    [shiv, hegde]|  hegde|
|shubh,g-hegde|  2|[shubh, g, hegde]|  shubh|
|shubh,g-hegde|  2|[shubh, g, hegde]|      g|
|shubh,g-hegde|  2|[shubh, g, h

In [275]:
transformed_df.select(date_format(transformed_df.sale_date, "MMMM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MMM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "MM")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "yyyy")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "yy")).show(1)
transformed_df.select(date_format(transformed_df.sale_date, "dd")).show(1)

+----------------------------+
|date_format(sale_date, MMMM)|
+----------------------------+
|                     January|
+----------------------------+
only showing top 1 row

+---------------------------+
|date_format(sale_date, MMM)|
+---------------------------+
|                        Jan|
+---------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, MM)|
+--------------------------+
|                        01|
+--------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, MM)|
+--------------------------+
|                        01|
+--------------------------+
only showing top 1 row

+----------------------------+
|date_format(sale_date, yyyy)|
+----------------------------+
|                        2024|
+----------------------------+
only showing top 1 row

+--------------------------+
|date_format(sale_date, yy)|
+--------------------------+
|                        24|
+-------------

#### Joins

In [301]:
data1 = [
    (1, "Chaitra", 2),
    (2, "Shiv", 2),
    (3, "Shubh", 3),
    (4, "Ram", 5),
    (5, "abc", 6)
]

data2 = [
    (1, "HR"),
    (2, "IT"),
    (3, "Finance"),
    (4, "Sales")
]

schema1 = ["emp_id", "name", "dept_id"]
schema2 = ["dept_id", "department"]

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

df1.show()
df2.show()

+------+-------+-------+
|emp_id|   name|dept_id|
+------+-------+-------+
|     1|Chaitra|      2|
|     2|   Shiv|      2|
|     3|  Shubh|      3|
|     4|    Ram|      5|
|     5|    abc|      6|
+------+-------+-------+

+-------+----------+
|dept_id|department|
+-------+----------+
|      1|        HR|
|      2|        IT|
|      3|   Finance|
|      4|     Sales|
+-------+----------+



In [302]:
df1.join(df2, on="dept_id", how="inner").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
+-------+------+-------+----------+



In [303]:
df1.join(df2, on="dept_id", how="left").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
|      5|     4|    Ram|      NULL|
|      6|     5|    abc|      NULL|
+-------+------+-------+----------+



In [304]:
df1.join(df2, on="dept_id", how="right").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      1|  NULL|   NULL|        HR|
|      2|     2|   Shiv|        IT|
|      2|     1|Chaitra|        IT|
|      3|     3|  Shubh|   Finance|
|      4|  NULL|   NULL|     Sales|
+-------+------+-------+----------+



In [305]:
df1.join(df2, on="dept_id", how="outer").show()

+-------+------+-------+----------+
|dept_id|emp_id|   name|department|
+-------+------+-------+----------+
|      1|  NULL|   NULL|        HR|
|      2|     1|Chaitra|        IT|
|      2|     2|   Shiv|        IT|
|      3|     3|  Shubh|   Finance|
|      4|  NULL|   NULL|     Sales|
|      5|     4|    Ram|      NULL|
|      6|     5|    abc|      NULL|
+-------+------+-------+----------+



In [306]:
df1.join(df2, on="dept_id", how="left_semi").show()  # Same as inner but includes columns only from left df

+-------+------+-------+
|dept_id|emp_id|   name|
+-------+------+-------+
|      2|     1|Chaitra|
|      2|     2|   Shiv|
|      3|     3|  Shubh|
+-------+------+-------+



In [288]:
df1.join(df2, on="dept_id", how="left_anti").show()  # df1 - df2 select * from left_table where ID not in (select ID from right_table)

+-------+------+----+
|dept_id|emp_id|name|
+-------+------+----+
|      5|     4| Ram|
|      6|     5| abc|
+-------+------+----+



In [289]:
df1.crossJoin(df2).show()  # Cartesian product

+------+-------+-------+-------+----------+
|emp_id|   name|dept_id|dept_id|department|
+------+-------+-------+-------+----------+
|     1|Chaitra|      2|      1|        HR|
|     1|Chaitra|      2|      2|        IT|
|     1|Chaitra|      2|      3|   Finance|
|     1|Chaitra|      2|      4|     Sales|
|     2|   Shiv|      2|      1|        HR|
|     2|   Shiv|      2|      2|        IT|
|     2|   Shiv|      2|      3|   Finance|
|     2|   Shiv|      2|      4|     Sales|
|     3|  Shubh|      3|      1|        HR|
|     3|  Shubh|      3|      2|        IT|
|     3|  Shubh|      3|      3|   Finance|
|     3|  Shubh|      3|      4|     Sales|
|     4|    Ram|      5|      1|        HR|
|     4|    Ram|      5|      2|        IT|
|     4|    Ram|      5|      3|   Finance|
|     4|    Ram|      5|      4|     Sales|
|     5|    abc|      6|      1|        HR|
|     5|    abc|      6|      2|        IT|
|     5|    abc|      6|      3|   Finance|
|     5|    abc|      6|      4|

#### Union VS UnionByName

In [296]:
data1 = [
    (1, "Chaitra"),
    (2, "Shiv"),
    (3, "Shubh"),
]

data2 = [
    ("Shiv", 2),
    ("Shubh", 3),
    ("abc", 4),
]

schema1 = ["emp_id", "name"]
schema2 = ["name", "emp_id"]

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

df1.show()
df2.show()
df1.union(df2).show()
df1.unionByName(df2).show()

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
+------+-------+

+-----+------+
| name|emp_id|
+-----+------+
| Shiv|     2|
|Shubh|     3|
|  abc|     4|
+-----+------+

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
|  Shiv|      2|
| Shubh|      3|
|   abc|      4|
+------+-------+

+------+-------+
|emp_id|   name|
+------+-------+
|     1|Chaitra|
|     2|   Shiv|
|     3|  Shubh|
|     2|   Shiv|
|     3|  Shubh|
|     4|    abc|
+------+-------+



In [299]:
df1.sort(['emp_id'], ascending=[0,1]).show()

+------+-------+
|emp_id|   name|
+------+-------+
|     3|  Shubh|
|     2|   Shiv|
|     1|Chaitra|
+------+-------+



#### Incremental Loading + Delta MERGE 

In [21]:
# CREATE TABLE src_emp (
#     emp_id INT PRIMARY KEY,
#     name VARCHAR(100) NOT NULL,
#     salary INT,
#     promoted BOOLEAN,
#     state VARCHAR(50),
#     date_modified TIMESTAMP
# );

# INSERT INTO src_emp (emp_id, name, salary, promoted, state, date_modified) VALUES
# (1, 'Ravi Kumar', 55000.00, 1, 'Karnataka', '2026-04-01'),
# (2, 'Anita Sharma', 62000.50, 0, 'Maharashtra', '2026-03-20'),
# (3, 'Suresh Reddy', 48000.75, 1, 'Telangana', '2024-01-01'),
# (4, 'Priya Nair', 71000.00, 1, 'Kerala', '2026-02-15'),
# (5, 'Amit Singh', 53000.25, 0, 'Delhi', '2022-06-01'),
# (6, 'Sneha Patil', 60000.00, 1, 'Maharashtra', '2026-04-05'),
# (7, 'Arjun Das', 45000.00, 0, 'West Bengal', '2026-03-30'),
# (8, 'Meera Iyer', 68000.90, 1, 'Tamil Nadu', '2026-01-10');

from delta.tables import DeltaTable
try:
    last_read_df = spark.read.format("delta").load("s3a://dbt-snow-bucket/emp")
    last_read = last_read_df.select(max(last_read_df.date_modified)).collect()[0][0]
    old_df = DeltaTable.forPath(spark, "s3a://dbt-snow-bucket/emp")
except Exception as e:
    print(e)
    last_read = '1900-01-01'
print(last_read)

2026-04-05 00:00:00


In [22]:
jdbc_url = "jdbc:mysql://mysql_container:3306/employees"
new_df = spark.read.format("jdbc").option("url", jdbc_url)\
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", f"""
        (SELECT * FROM src_emp 
         WHERE date_modified > '{last_read}') AS inc """)\
    .option("user", "root") \
    .option("password", "root").load()
new_df.show()

+------+----------+------+--------+---------+-------------------+
|emp_id|      name|salary|promoted|    state|      date_modified|
+------+----------+------+--------+---------+-------------------+
|     1|Ravi Kumar| 55000|   false|Karnataka|2026-04-06 00:00:00|
|     9|     Hegde|255000|    true|Karnataka|2026-04-06 00:00:00|
+------+----------+------+--------+---------+-------------------+



In [25]:
new_df = new_df.withColumn("start_date", current_date())\
    .withColumn("end_date", lit(None).cast(DateType())) \
    .withColumn("is_current", lit(True))

if last_read == "1900-01-01":
    new_df.write.format("delta").mode("append").save("s3a://dbt-snow-bucket/emp")

else:
    old_df.alias("old").merge(
        new_df.alias("new"),
        "old.emp_id = new.emp_id AND old.is_current = true"
    ).whenMatchedUpdate(
        condition="old.promoted <> new.promoted",
        set={
            "end_date": "current_date()",
            "is_current": "false"
        }
    ).execute()

    # Insert new version of changed records
    new_records = new_df.withColumn("start_date", current_date()) \
                       .withColumn("end_date", lit(None)) \
                       .withColumn("is_current", lit(True))
    
    old_df.alias("old").merge(
        new_records.alias("new"),
        "old.emp_id = new.emp_id AND old.is_current = true"
    ).whenNotMatchedInsertAll().execute()


In [3]:
# After seconf load
last_read_df = spark.read.format("delta").load("s3a://dbt-snow-bucket/emp")
last_read_df.show()

+------+------------+------+--------+-----------+-------------------+----------+----------+----------+
|emp_id|        name|salary|promoted|      state|      date_modified|start_date|  end_date|is_current|
+------+------------+------+--------+-----------+-------------------+----------+----------+----------+
|     1|  Ravi Kumar| 55000|    true|  Karnataka|2026-04-01 00:00:00|2026-04-21|2026-04-21|     false|
|     6| Sneha Patil| 60000|    true|Maharashtra|2026-04-05 00:00:00|2026-04-21|      NULL|      true|
|     3|Suresh Reddy| 48001|    true|  Telangana|2024-01-01 00:00:00|2026-04-21|      NULL|      true|
|     5|  Amit Singh| 53000|   false|      Delhi|2022-06-01 00:00:00|2026-04-21|      NULL|      true|
|     4|  Priya Nair| 71000|    true|     Kerala|2026-02-15 00:00:00|2026-04-21|      NULL|      true|
|     8|  Meera Iyer| 68001|    true| Tamil Nadu|2026-01-10 00:00:00|2026-04-21|      NULL|      true|
|     7|   Arjun Das| 45000|   false|West Bengal|2026-03-30 00:00:00|2026

In [17]:
# After First load
last_read_df = spark.read.format("delta").load("s3a://dbt-snow-bucket/emp")
last_read_df.show()

+------+------------+------+--------+-----------+-------------------+----------+--------+----------+
|emp_id|        name|salary|promoted|      state|      date_modified|start_date|end_date|is_current|
+------+------------+------+--------+-----------+-------------------+----------+--------+----------+
|     1|  Ravi Kumar| 55000|    true|  Karnataka|2026-04-01 00:00:00|2026-04-21|    NULL|      true|
|     2|Anita Sharma| 62001|   false|Maharashtra|2026-03-20 00:00:00|2026-04-21|    NULL|      true|
|     3|Suresh Reddy| 48001|    true|  Telangana|2024-01-01 00:00:00|2026-04-21|    NULL|      true|
|     4|  Priya Nair| 71000|    true|     Kerala|2026-02-15 00:00:00|2026-04-21|    NULL|      true|
|     5|  Amit Singh| 53000|   false|      Delhi|2022-06-01 00:00:00|2026-04-21|    NULL|      true|
|     6| Sneha Patil| 60000|    true|Maharashtra|2026-04-05 00:00:00|2026-04-21|    NULL|      true|
|     7|   Arjun Das| 45000|   false|West Bengal|2026-03-30 00:00:00|2026-04-21|    NULL|  

In [13]:
Z order demo
data cleaning 
data validation
parity checks


```sql
SELECT user_id, COUNT(*) AS orders
FROM orders
WHERE status = 'completed'
GROUP BY user_id;

| Concept        | Description                     |
|----------------|---------------------------------|
| DBT            | Data transformation tool        |
| Model          | SQL file in DBT                 |
| Materialization| Table/View creation strategy    |

<details>
<summary><b>Click to expand</b></summary>

Your notes go here

- Point 1  
- Point 2  

</details>